### Imports

In [1]:
# Import des outils de tables
import pandas as pd
import kagglehub

# Import des outils de machine learning
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV

# Import des modèles
from tabicl import TabICLClassifier # https://github.com/soda-inria/tabicl
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Import des métriques d'évaluation
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

### Téléchargement du dataset

In [2]:
df = kagglehub.dataset_load(
    kagglehub.KaggleDatasetAdapter.PANDAS,
    "heptapod/titanic",
    "train_and_test2.csv"
)

df.head()

,Passengerid,Age,Fare,Sex,sibsp,zero,zero.1,zero.2,zero.3,zero.4,...,zero.12,zero.13,zero.14,Pclass,zero.15,zero.16,Embarked,zero.17,zero.18,2urvived
0,1,22.0,7.2500,0,1,0,0,0,0,0,...,0,0,0,3,0,0,2.0,0,0,0
1,2,38.0,71.2833,1,1,0,0,0,0,0,...,0,0,0,1,0,0,0.0,0,0,1
2,3,26.0,7.9250,1,0,0,0,0,0,0,...,0,0,0,3,0,0,2.0,0,0,1
3,4,35.0,53.1000,1,1,0,0,0,0,0,...,0,0,0,1,0,0,2.0,0,0,1
4,5,35.0,8.0500,0,0,0,0,0,0,0,...,0,0,0,3,0,0,2.0,0,0,0


### Séparation en train, test

In [3]:
# Variables explicatives
X = df.iloc[:,:-1]

# Variable à prédire
y = df.iloc[:, -1] # df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(1047, 27)
(262, 27)
(1047,)
(262,)


### Prédiction avec TFM ( Tabular Foundation Model )

In [4]:
model_tfm = TabICLClassifier(kv_cache=True)
model_tfm.fit(X_train, y_train)

y_pred_clf = model_tfm.predict(X_test)

### Prédiction avec Random Forest

In [5]:
model_rf = RandomForestClassifier(random_state=42)
model_rf.fit(X_train, y_train)

y_pred_rf = model_rf.predict(X_test)

### Prédiction avec XGBoost

In [6]:
param_grid = {
    "n_estimators": [100, 300, 500],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [3, 4, 5],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1]
}

grid_xgb = GridSearchCV(
    XGBClassifier(
        random_state=42,
        eval_metric="logloss"
    ),
    param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

grid_xgb.fit(X_train, y_train)

y_pred_xgb = grid_xgb.predict(X_test)

print(grid_xgb.best_params_)

{'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 100, 'subsample': 0.8}


### Évaluation des résultats

In [7]:
# ------------------------------------------------------------
# Métriques du TFM (Tabular Foundation Model - TabICL)
# Prédictions stockées dans : y_pred_clf
# ------------------------------------------------------------

accuracy_tfm = accuracy_score(y_test, y_pred_clf)
precision_tfm = precision_score(y_test, y_pred_clf)
recall_tfm = recall_score(y_test, y_pred_clf)
f1_tfm = f1_score(y_test, y_pred_clf)


# ------------------------------------------------------------
# Métriques du Random Forest
# Prédictions stockées dans : y_pred_rf
# ------------------------------------------------------------

accuracy_rf = accuracy_score(y_test, y_pred_rf)
precision_rf = precision_score(y_test, y_pred_rf)
recall_rf = recall_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf)

# ------------------------------------------------------------
# Métriques du XGBoost
# Prédictions stockées dans : y_pred_xgb
# ------------------------------------------------------------

accuracy_xgb = accuracy_score(y_test, y_pred_xgb)
precision_xgb = precision_score(y_test, y_pred_xgb)
recall_xgb = recall_score(y_test, y_pred_xgb)
f1_xgb = f1_score(y_test, y_pred_xgb)

### Tableau Comparatif

In [8]:
resultats = pd.DataFrame({

    "Métrique": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score"
    ],

    "TFM (TabICL)": [
        accuracy_tfm,
        precision_tfm,
        recall_tfm,
        f1_tfm
    ],

    "Random Forest": [
        accuracy_rf,
        precision_rf,
        recall_rf,
        f1_rf
    ],

    "XGBoost": [
        accuracy_xgb,
        precision_xgb,
        recall_xgb,
        f1_xgb
    ],

    "Description": [
        "Accuracy : plus proche de 1 = meilleur",
        "Precision : plus proche de 1 = meilleur",
        "Recall : plus proche de 1 = meilleur",
        "F1-score : plus proche de 1 = meilleur"
    ]
})

# Formate les scores pour faciliter la lecture
def format_nombre(x):
    if abs(x) >= 1:
        return f"{x:_.0f}".replace("_", " ")
    else:
        return f"{x:.3f}"

resultats["TFM (TabICL)"] = resultats["TFM (TabICL)"].apply(format_nombre)
resultats["Random Forest"] = resultats["Random Forest"].apply(format_nombre)
resultats["XGBoost"] = resultats["XGBoost"].apply(format_nombre)

# Affichage du tableau comparatif
resultats

,Métrique,TFM (TabICL),Random Forest,XGBoost,Description
0,Accuracy,0.889,0.859,0.874,Accuracy : plus proche de 1 = meilleur
1,Precision,0.844,0.781,0.794,Precision : plus proche de 1 = meilleur
2,Recall,0.740,0.685,0.740,Recall : plus proche de 1 = meilleur
3,F1-score,0.788,0.730,0.766,F1-score : plus proche de 1 = meilleur
